<a href="https://colab.research.google.com/github/josephgalicinao/SkinLesionDetection/blob/main/Skin_Lesion_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

Important imports and packages used

In [1]:
from google.colab import userdata
import os
import kagglehub
import pandas as pd
import tensorflow as tf
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold

import csv
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

IMG_SIZE   = (128, 128)
AUTOTUNE   = tf.data.AUTOTUNE

label_map = {'Benign': 0,
             'Malignant': 1}

## Load Dataset

In [2]:
os.environ["KAGGLE_USERNAME"] =  userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle

ISIC 2018

In [3]:
isic2018_path = kagglehub.dataset_download("josephgalicinao/isic-2018-dataset")

print("Path to dataset files:", isic2018_path)

100%|██████████| 237M/237M [00:02<00:00, 115MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-dataset/versions/1


ISIC 2018 Test

In [4]:
# Download latest version
isic2018_test_path = kagglehub.dataset_download("josephgalicinao/isic-2018-test-dataset")

print("Path to dataset files:", isic2018_test_path)

100%|██████████| 37.3M/37.3M [00:00<00:00, 93.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-test-dataset/versions/1


## Create Datasets

Create Fitzpatrick dataset + Unlabeled Fitzpatrick

In [5]:
def get_train_isic2018():
  print("ISIC 2018 Training Dataset...")
  df = pd.read_csv(f"{isic2018_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values
  isic_patient_ids = df["lesion_id"].values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx) == len(isic_patient_ids)

  return isic_paths, isic_dx, isic_patient_ids

def get_test_isic2018():
  print("ISIC 2018 Test Dataset...")
  df = pd.read_csv(f"{isic2018_test_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_test_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx)

  return isic_paths, isic_dx

# Feature Extraction

## Helper Functions

Functions calculate IoU, lesion border, area, and perimeter

In [6]:
def calculate_iou(mask, groundtruth_mask):
    mask = (mask > 0).astype(np.uint8)  # Now contains 0s and 1s
    groundtruth_mask = (groundtruth_mask > 0).astype(np.uint8)

    overlap = np.logical_and(mask, groundtruth_mask)
    union = np.logical_or(mask, groundtruth_mask)

    iou = np.sum(overlap) / np.sum(union)

    return iou

def get_max_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    max_contour = max(contours, key=cv2.contourArea)
    return max_contour

def get_contour_area(contour):
    return cv2.contourArea(contour)

def get_contour_perimeter(contour):
    return cv2.arcLength(contour, True)

## Asymmetry

Roundness

In [9]:
def roundness(area, perimeter):
    roundness_score = (4 * np.pi  * area) / (perimeter * perimeter)
    return roundness_score

Symmetry Distance

In [10]:
def symmetry_distance(mask, contour, debug):
    n = len(contour)

    # Separate the x and y coords
    points = contour.reshape(-1, 2)
    x_coords = points[:, 0]
    y_coords = points[:, 1]

    # Calculate the centroid
    M = cv2.moments(contour)
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    center = np.array([cx, cy])

    # 1) Folding
    folded_x_coords = []
    folded_y_coords = []
    i = 0
    for x_coord, y_coord in zip(x_coords, y_coords):
        theta = (2 * np.pi * i) / n
        folded_x = center[0] + np.cos(theta) * (x_coord - center[0]) - np.sin(theta) * (y_coord - center[1])
        folded_y = center[1] + np.sin(theta) * (x_coord - center[0]) + np.cos(theta) * (y_coord - center[1])
        folded_x_coords.append(folded_x)
        folded_y_coords.append(folded_y)
        i += 1

    folded_x_coords = np.array(folded_x_coords, np.float32)
    folded_y_coords = np.array(folded_y_coords, np.float32)

    # 2) Plot P0
    p0_x = np.mean(folded_x_coords)
    p0_y = np.mean(folded_y_coords)

    # 3) Unfold
    unfolded_x_coords = []
    unfolded_y_coords = []
    i = 0
    while i < n:
        theta = (-2 * np.pi * i) / n
        unfolded_x = center[0] + np.cos(theta) * (p0_x - center[0]) - np.sin(theta) * (p0_y - center[1])
        unfolded_y = center[1] + np.sin(theta) * (p0_x - center[0]) + np.cos(theta) * (p0_y - center[1])
        unfolded_x_coords.append(unfolded_x)
        unfolded_y_coords.append(unfolded_y)
        i += 1

    unfolded_x_coords = np.array(unfolded_x_coords, np.float32)
    unfolded_y_coords = np.array(unfolded_y_coords, np.float32)

    # 4) Calcualte the symmetry distance
    sd = np.sum((x_coords - unfolded_x_coords) ** 2 + (y_coords - unfolded_y_coords) ** 2) / n

    if debug:
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(x_coords, y_coords, 'go-', markersize=1)  # Connects in order
        ax.plot([x_coords[-1], x_coords[0]], [y_coords[-1], y_coords[0]], 'go-', label="Contour", markersize=1)  # Closes the triangle manually

        ax.plot(folded_x_coords, folded_y_coords, 'bo-', markersize=1)  # Connects in order
        ax.plot([folded_x_coords[-1], folded_x_coords[0]], [folded_y_coords[-1], folded_y_coords[0]], 'bo-', label="Folded Shape", markersize=1)  # Closes the triangle manually

        ax.plot(unfolded_x_coords, unfolded_y_coords, 'mo-', markersize=1)  # Connects in order
        ax.plot([unfolded_x_coords[-1], unfolded_x_coords[0]], [unfolded_y_coords[-1], unfolded_y_coords[0]], 'mo-', label="Unfolded Shape", markersize=1)  # Closes the triangle manually

        ax.plot(p0_x, p0_y, '*', label="P0", markersize=8, color="gold")

        ax.set_xlim(0, 600)
        ax.set_ylim(450, 0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.legend(loc="upper right")

        plt.tight_layout()
        plt.show()

    return sd

Reflection Across Principle Axis

In [11]:
def rotated_image(theta, x, y):
    # Principal Axis Rotation
    rotated_x = np.cos(-theta) * (x) - np.sin(-theta) * (y)
    rotated_y = np.sin(-theta) * (x) + np.cos(-theta) * (y)

    x_min = rotated_x.min()
    y_min = rotated_y.min()

    bounded_x = (rotated_x - x_min).astype(np.int32)
    bounded_y = (rotated_y - y_min).astype(np.int32)

    new_contour = np.stack((bounded_x, bounded_y), axis=1)
    x_min, x_max = bounded_x.min(), bounded_x.max()
    y_min, y_max = bounded_y.min(), bounded_y.max()

    new_mask = np.zeros(shape=(y_max - y_min, x_max - x_min))  # single-channel
    cv2.drawContours(new_mask, [new_contour], -1, color=255, thickness=cv2.FILLED)

    return new_mask

def principle_axis(mask, contour):

    # Get the x and y coords
    points = contour.reshape(-1, 2)
    x_coords = points[:, 0]
    y_coords = points[:, 1]

    # Calculate the center of the centroid
    M = cv2.moments(contour)
    cx, cy = M['m10']/M['m00'], M['m01']/M['m00']

    # Calculate the orientation of the major axis relative to the horizontal axis
    Ixx, Iyy, Ixy = M['mu20'], M['mu02'], M['mu11']
    major_axis_theta = 0.5 * np.arctan2(2.0 * Ixy, Ixx - Iyy)   # radians

    # Center lesion at (0, 0)
    translated_x = x_coords - cx
    translated_y = y_coords - cy

    rotated_mask = rotated_image(major_axis_theta, translated_x, translated_y)

    major_flipped_mask = cv2.flip(rotated_mask, 0)
    minor_flipped_mask = cv2.flip(rotated_mask, 1)

    major_axis_iou = calculate_iou(major_flipped_mask, rotated_mask)
    minor_axis_iou = calculate_iou(minor_flipped_mask, rotated_mask)

    return major_axis_iou, minor_axis_iou

## Border Irregularities

Minimum Enscribed Circle

In [12]:
def best_fit_circle(mask, contour, debug):

    # Fits the best ellipse to the contour
    center, radius = cv2.minEnclosingCircle(contour)

    circle_area = np.pi * (radius * radius)
    contour_area = cv2.contourArea(contour)
    circle_area_ratio = abs(1.0 - circle_area / contour_area)

    # Calculates assymmetry score based on iou
    h, w = mask.shape[:2]
    filled_contour = np.zeros((h, w), dtype=np.uint8)
    filled_circle = np.zeros((h, w), dtype=np.uint8)

    cv2.drawContours(filled_contour, [contour], 0, 255, -1)
    cv2.circle(filled_circle, (int(center[0]), int(center[1])), int(radius), 255, thickness=-1)
    circle_iou = calculate_iou(filled_contour, filled_circle)

    if debug:
        # Print the feature values
        print(f"Circle Area Ratio: {circle_area_ratio}")
        print(f"Circle IoU: {circle_iou}")
        # --- Fit minimum enclosing circle ---
        fig, ax1 = plt.subplots(figsize=(10, 5))

        t = np.linspace(0, 2*np.pi, 300)

        (center_x, center_y), radius = cv2.minEnclosingCircle(contour)
        x_circle = center_x + radius * np.cos(t)
        y_circle = center_y + radius * np.sin(t)

        # Right: circle
        ax1.imshow(mask)
        ax1.plot(x_circle, y_circle, 'b-', linewidth=2, label='Min Enclosing Circle')
        ax1.scatter(center_x, center_y, marker='o', s=100, color='blue', label='Circle Center')
        ax1.axis('off')
        ax1.set_title('Minimum Enclosing Circle')
        ax1.legend()

        plt.tight_layout()
        plt.show()


    return circle_area_ratio, circle_iou

Convex Hull

In [13]:
def convex_hull(mask, contour, debug):
    # Get the convex hull
    hull = cv2.convexHull(contour)

    # Get the hull area and the contour area
    hull_area = cv2.contourArea(hull)
    contour_area = cv2.contourArea(contour)
    convex_area_ratio = abs(1.0 - hull_area / contour_area)

    h, w = mask.shape[:2]
    filled_contour = np.zeros((h, w), dtype=np.uint8)
    filled_convex = np.zeros((h, w), dtype=np.uint8)

    cv2.drawContours(filled_contour, [contour], 0, 255, -1)
    cv2.drawContours(filled_convex, [hull], 0, 255, -1)
    convex_iou = calculate_iou(filled_contour, filled_convex)

    if debug:
        print(f"Convex Area Ratio: {convex_area_ratio}")
        print(f"Convex IoU: {convex_iou}")

        plt.imshow(mask)
        plt.plot(contour[:,0,0], contour[:,0,1], 'g-', label='Contour')  # green
        hull_points = hull[:,0,:]
        plt.plot(np.append(hull_points[:,0], hull_points[0,0]),
                np.append(hull_points[:,1], hull_points[0,1]), 'r-', label='Convex Hull')  # red
        plt.legend()
        plt.axis("off")
        plt.show()

    return convex_area_ratio, convex_iou

## Color Features

Color Entropy

In [14]:
from skimage.measure import shannon_entropy

def color_entropy(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV_FULL)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_entropy = shannon_entropy((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_entropy = shannon_entropy(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_entropy = shannon_entropy(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_entropy, ch2_entropy, ch3_entropy

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_entropy = shannon_entropy(skin_lesion_image[:, 0].flatten())
    ch2_entropy = shannon_entropy(skin_lesion_image[:, 1].flatten())
    ch3_entropy = shannon_entropy(skin_lesion_image[:, 2].flatten())

    return ch1_entropy, ch2_entropy, ch3_entropy

Color Standard Deviation

In [15]:
def color_std(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV_FULL)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_std = (127.5 * (np.sin(scaled_input) + 1)).flatten().std()
        ch2_std = hsv_image[:, 1].astype(np.float64).flatten().std()
        ch3_std = hsv_image[:, 2].astype(np.float64).flatten().std()

        return ch1_std, ch2_std, ch3_std

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_std = skin_lesion_image[:, 0].flatten().std()
    ch2_std = skin_lesion_image[:, 1].flatten().std()
    ch3_std = skin_lesion_image[:, 2].flatten().std()

    return ch1_std, ch2_std, ch3_std

Color Skew

In [16]:
from scipy.stats import skew

def color_skew(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, color_space)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_skew = skew((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_skew = skew(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_skew = skew(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_skew, ch2_skew, ch3_skew

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_skew = skew(skin_lesion_image[:, 0].flatten())
    ch2_skew = skew(skin_lesion_image[:, 1].flatten())
    ch3_skew = skew(skin_lesion_image[:, 2].flatten())

    return ch1_skew, ch2_skew, ch3_skew

Color Mean

In [19]:
def color_mean(image, mask, color_space):
  if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, color_space)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_mean = np.mean((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_mean = np.mean(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_mean = np.mean(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_mean, ch2_mean, ch3_mean

  image = cv2.cvtColor(image, color_space)
  skin_lesion_image = image[mask == 255]
  ch1_mean = np.mean(skin_lesion_image[:, 0].flatten())
  ch2_mean = np.mean(skin_lesion_image[:, 1].flatten())
  ch3_mean = np.mean(skin_lesion_image[:, 2].flatten())

  return ch1_mean, ch2_mean, ch3_mean

# Feature Extraction

### Load Dataset

In [20]:
IMG_SIZE   = (128, 128)
AUTOTUNE   = tf.data.AUTOTUNE

def segmentation_load_dataset(img_path, y):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)   # or decode_png
    img = tf.image.resize(img, (128, 128))
    img = tf.cast(img, tf.float32) / 255.0

    return img, y

## Loss Functions

In [21]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    sums = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (sums + smooth)
    return 1.0 - tf.reduce_mean(dice)

def iou_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])  # sum over H, W, C
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(iou)  # average IoU across batch

def bce_iou_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    iou = iou_loss(y_true, y_pred)
    beta = 0.25
    return beta * bce + (1 - beta) * iou

def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    dice = dice_loss(y_true, y_pred)
    return bce + tf.math.log(tf.math.cosh(dice))

## Evaluation Metrics

In [22]:
def hard_dice(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3])

    # Dice coefficient per image
    dice = (2 * intersection + smooth) / (union + smooth)

    # Return mean Dice over batch
    return tf.reduce_mean(dice)

def hard_iou(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection

    # IoU per image
    iou = (intersection + smooth) / (union + smooth)

    # Return mean IoU over batch
    return tf.reduce_mean(iou)

## Extract features

In [23]:
def get_features(skin_lesion_image, mask):

  try:
    mask = mask.astype(np.uint8)

    # Basic lesion features
    contour = get_max_contour(mask=mask)
    perimeter = get_contour_perimeter(contour)
    area = get_contour_area(contour)

    # -----Asymmetry-----
    circularity = roundness(area, perimeter)
    sd = symmetry_distance(mask, contour, debug=False)
    major_axis_iou, minor_axis_iou = principle_axis(mask, contour)

    # -----Border-----
    circle_iou, circle_area_ratio = best_fit_circle(mask, contour, debug=False)
    convex_hull_iou, convex_hull_area_ratio = convex_hull(mask, contour, debug=False)

    # -----Color-----
    red_std, green_std, blue_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_std, a_std, b_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_std, cr_std, cb_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_std, s_std, v_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_entropy, green_entropy, blue_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_entropy, a_entropy, b_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_entropy, cr_entropy, cb_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_entropy, s_entropy, v_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_skew, green_skew, blue_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_skew, a_skew, b_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_skew, cr_skew, cb_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_skew, s_skew, v_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_mean, green_mean, blue_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_mean, a_mean, b_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_mean, cr_mean, cb_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_mean, s_mean, v_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    features = [circularity, sd, major_axis_iou, minor_axis_iou,
                circle_iou, circle_area_ratio, convex_hull_iou, convex_hull_area_ratio,
                red_std, green_std, blue_std, l_std, a_std, b_std, y_std, cr_std, cb_std, h_std, s_std, v_std,
                red_entropy, green_entropy, blue_entropy, l_entropy, a_entropy, b_entropy, y_entropy, cr_entropy, cb_entropy, h_entropy, s_entropy, v_entropy,
                red_skew, green_skew, blue_skew, l_skew, a_skew, b_skew, y_skew, cr_skew, cb_skew, h_skew, s_skew, v_skew,
                red_mean, green_mean, blue_mean, l_mean, a_mean, b_mean, y_mean, cr_mean, cb_mean, h_mean, s_mean, v_mean]

    return features
  except:
    return [0] * 56

Get ABCDE features

In [ ]:
# Combined dataset

batch = 64
train_imgs, train_dx, train_patient_id = get_train_isic2018()
val_imgs, val_dx = get_test_isic2018()

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_imgs, train_dx))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_dx))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

batch = 64

segmentation_model_path = f'/content/drive/MyDrive/Thesis/Segmentation/segmentation_model.keras'
segmentation_model = tf.keras.models.load_model(segmentation_model_path,
                                                  custom_objects={"bce_iou_loss": bce_iou_loss,
                                                                  "hard_dice": hard_dice,
                                                                  "hard_iou": hard_iou})

with (open(f'/content/drive/MyDrive/Thesis/abcde_features_train.csv', 'w', newline='') as csvfile1):
  csv_writer1 = csv.writer(csvfile1)

  training_masks = segmentation_model.predict(train_ds)

  for i in tqdm(range(len(train_imgs)),
                      total=len(train_imgs),
                      desc="Getting automatic training ABCDE train features"):

    cur_mask = training_masks["o1"][i]
    cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255
    cur_mask = cv2.resize(cur_mask, (600, 450))
    cur_image = cv2.imread(train_imgs[i])


    features = get_features(cur_image, cur_mask)

    csv_writer1.writerow([train_imgs[i], train_dx[i], train_patient_id[i]] + features)

with (open(f'/content/drive/MyDrive/Thesis/abcde_features_test.csv', 'w', newline='') as csvfile1):
  csv_writer1 = csv.writer(csvfile1)

  val_masks = segmentation_model.predict(val_ds)

  for i in tqdm(range(len(val_imgs)),
                      total=len(val_imgs),
                      desc="Getting automatic training ABCDE train features"):

    cur_mask = val_masks["o1"][i]
    cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255
    cur_mask = cv2.resize(cur_mask, (600, 450))
    cur_image = cv2.imread(val_imgs[i])

    features = get_features(cur_image, cur_mask)

    csv_writer1.writerow([val_imgs[i], val_dx[i]] + features)

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...
157/157 ━━━━━━━━━━━━━━━━━━━━ 106s 480ms/step


Getting automatic training ABCDE train features:  24%|██▍       | 2436/10015 [06:20<23:24,  5.39it/s]